# Knot rare-event benchmark — main reproduction notebook

This notebook mirrors the script-based reproduction pipeline described in the repository README.

**Canonical reproduction path:** the scripts in `scripts/` remain the official entry points.  
**Notebook purpose:** interactive inspection, quick reproduction of main tables, and optional appendix diagnostics.

The notebook assumes it is being run from the repository root.

## 0. Setup

Create output folders and configure optional long-running sections.

By default, the notebook runs the core pipeline and the most important diagnostics. Longer appendix experiments can be enabled through flags.

In [ ]:
import sys
from pathlib import Path

# Run from repository root.
sys.path.append(".")

import numpy as np
import pandas as pd

# Display helper for notebooks
try:
    from IPython.display import display, Image
except Exception:
    display = print

# Output directories consistent with README
RAW_DIR = Path("data/raw")
OUT_TABLES = Path("results/tables")
OUT_FIGS = Path("results/figures")
OUT_SCORES = Path("results/scores")
OUT_MODELS = Path("results/models")
OUT_SPLITS = Path("results/splits")

for p in [OUT_TABLES, OUT_FIGS, OUT_SCORES, OUT_MODELS, OUT_SPLITS]:
    p.mkdir(parents=True, exist_ok=True)

# Optional long-running sections
RUN_LONG_ABLATION = False
RUN_DISTRIBUTION_DIAGNOSTICS = False
RUN_SUPPORT_WIDTH_STRATIFICATION = False
RUN_OOD_PLOT = False

SEED = 42

## 1. Load raw data

Place the raw files in `data/raw/`:

```text
data/raw/Alexander_upto_17.csv
data/raw/Jones_upto_15_MIRRORS.csv
data/raw/HomflyPt_upto_15_MIRRORS.csv
```

For Colab, you may mount Drive and redefine `RAW_DIR`, but the default path follows the README.

In [ ]:
# Optional Colab usage:
# from google.colab import drive
# drive.mount("/content/drive")
# RAW_DIR = Path("/content/drive/MyDrive/Colab Notebooks")

A_path = RAW_DIR / "Alexander_upto_17.csv"
J_path = RAW_DIR / "Jones_upto_15_MIRRORS.csv"
H_path = RAW_DIR / "HomflyPt_upto_15_MIRRORS.csv"

for p in [A_path, J_path, H_path]:
    if not p.exists():
        raise FileNotFoundError(
            f"Missing raw file: {p}\n"
            "Place the raw Zenodo CSV files in data/raw/ or redefine RAW_DIR."
        )

A = pd.read_csv(A_path)
J = pd.read_csv(J_path)
H = pd.read_csv(H_path)

print("Loaded:")
print("A:", A.shape)
print("J:", J.shape)
print("H:", H.shape)

## 2. Preprocess, align, and split

This performs mirror filtering, alignment across invariants, signature consistency filtering, and fixed train/validation/test splitting.

All standardization is fit on the training split only.

In [ ]:
from src.data.preprocess import preprocess_align
from src.data.splits import prepare_splits_and_scalers

aligned = preprocess_align(
    A,
    J,
    H,
    max_cross=15,
    drop_mirrors=True,
    drop_signature_mismatches=True,
    verbose=True,
)

prepared = prepare_splits_and_scalers(
    aligned["X_A"],
    aligned["X_J"],
    aligned["X_H"],
    aligned["metadata"],
    seed=SEED,
)

A_tr = prepared["A"]["train"]
A_va = prepared["A"]["val"]
A_te = prepared["A"]["test"]

J_tr = prepared["J"]["train"]
J_va = prepared["J"]["val"]
J_te = prepared["J"]["test"]

H_tr = prepared["H"]["train"]
H_va = prepared["H"]["val"]
H_te = prepared["H"]["test"]

metadata_train = prepared["metadata"]["train"]
metadata_val = prepared["metadata"]["val"]
metadata_test = prepared["metadata"]["test"]

train_idx = prepared["indices"]["train"]
val_idx = prepared["indices"]["val"]
test_idx = prepared["indices"]["test"]

print("Aligned N before signature class filter:", aligned["debug"]["aligned_rows_final"])
print("Final N after signature class filter:", len(prepared["metadata"]["all"]))
print("Dropped signature classes:", prepared["dropped_signature_classes"])

print("A:", A_tr.shape, A_va.shape, A_te.shape)
print("J:", J_tr.shape, J_va.shape, J_te.shape)
print("H:", H_tr.shape, H_va.shape, H_te.shape)

print("\nTest signature counts:")
print(metadata_test["signature"].value_counts().sort_index())

# Save a compact preprocessing summary
summary_rows = []
for k, v in aligned["debug"].items():
    summary_rows.append({"key": k, "value": str(v)})
summary_rows.append({"key": "final_split_N", "value": str(len(prepared["metadata"]["all"]))})
summary_rows.append({"key": "dropped_signature_classes", "value": str(prepared["dropped_signature_classes"])})
pd.DataFrame(summary_rows).to_csv(OUT_TABLES / "preprocessing_summary.csv", index=False)

# Save split indices
np.savez(
    OUT_SPLITS / "split_indices_seed42.npz",
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx,
)

## 3. Task A: raw coefficient bulk decoding

This evaluates class-balanced multinomial logistic regression probes on raw coefficient vectors.

The knot signature is used only for this post hoc probe and is not used to train PCA or autoencoders.

In [ ]:
from src.evaluation.bulk_decoding import eval_multinomial_lr_split, majority_baseline
from src.features.summary_features import all_summary_features

y_train = metadata_train["signature"].to_numpy().astype(int)
y_val = metadata_val["signature"].to_numpy().astype(int)
y_test = metadata_test["signature"].to_numpy().astype(int)

raw_results = {
    "Jones": eval_multinomial_lr_split(J_tr, y_train, J_te, y_test),
    "Alexander": eval_multinomial_lr_split(A_tr, y_train, A_te, y_test),
    "HOMFLY": eval_multinomial_lr_split(H_tr, y_train, H_te, y_test),
}

raw_rows = []
for name, res in raw_results.items():
    print(f"{name:9s} Acc={res['accuracy']:.3f} Macro-F1={res['macro_f1']:.3f}")
    raw_rows.append({
        "Invariant": name,
        "Representation": "Raw coefficients",
        "Accuracy": res["accuracy"],
        "MacroF1": res["macro_f1"],
        "n_iter": res["n_iter"],
    })

majority = majority_baseline(y_train, y_test)
print("Majority baseline:", majority)

raw_probe_df = pd.DataFrame(raw_rows)
raw_probe_df.to_csv(OUT_TABLES / "bulk_raw_coefficients.csv", index=False)
display(raw_probe_df)

# Example: summary/confounder-only features on HOMFLY
Xtr_conf = all_summary_features(H_tr)
Xte_conf = all_summary_features(H_te)

conf_res = eval_multinomial_lr_split(Xtr_conf, y_train, Xte_conf, y_test)
print("HOMFLY summary/confounder-only:", conf_res["accuracy"], conf_res["macro_f1"])

## 4. PCA and autoencoder reconstruction models

This section fits PCA and autoencoders on the training split only, computes reconstruction scores on the test split, and evaluates PCA/AE latent probes.

Autoencoders are trained without signature labels.

In [ ]:
from sklearn.decomposition import PCA

from src.models.autoencoder import train_autoencoder, encode_with_model, set_global_seed
from src.evaluation.reconstruction_scores import nre_from_pca, nre_from_model
from src.evaluation.bulk_decoding import eval_multinomial_lr_split

set_global_seed(SEED)

X_store = {
    "Alexander": {"train": A_tr, "val": A_va, "test": A_te},
    "Jones": {"train": J_tr, "val": J_va, "test": J_te},
    "HOMFLY": {"train": H_tr, "val": H_va, "test": H_te},
}

ARCH = {
    "Alexander": {"latent": 16, "widths": [64, 32, 32]},
    "Jones": {"latent": 16, "widths": [128, 64, 64]},
    "HOMFLY": {"latent": 16, "widths": [256, 128, 128]},
}

D_PCA = 16

ae_models = {}
encoders = {}
pca_models = {}

scores_all = {}
latent_probe_rows = []

for inv in ["Alexander", "Jones", "HOMFLY"]:
    print(f"\n=== {inv} ===")

    Xtr = np.asarray(X_store[inv]["train"], dtype=np.float32)
    Xva = np.asarray(X_store[inv]["val"], dtype=np.float32)
    Xte = np.asarray(X_store[inv]["test"], dtype=np.float32)

    scores_all[inv] = {"PCA": {}, "AE": {}}

    # PCA fit on train only
    print("Fitting PCA...")
    pca = PCA(n_components=D_PCA, random_state=SEED)
    Z_pca_tr = pca.fit_transform(Xtr).astype(np.float32)
    Z_pca_te = pca.transform(Xte).astype(np.float32)

    pca_models[inv] = pca
    scores_all[inv]["PCA"]["test"] = nre_from_pca(pca, Xte)

    pca_probe = eval_multinomial_lr_split(
        Z_pca_tr,
        y_train,
        Z_pca_te,
        y_test,
        seed=SEED,
    )

    latent_probe_rows.append({
        "Invariant": inv,
        "Representation": f"PCA_d{D_PCA}",
        "Accuracy": pca_probe["accuracy"],
        "MacroF1": pca_probe["macro_f1"],
        "n_iter": pca_probe["n_iter"],
    })

    print(
        f"PCA probe: Acc={pca_probe['accuracy']:.3f}, "
        f"Macro-F1={pca_probe['macro_f1']:.3f}"
    )

    # Autoencoder train on train, early stopping on val
    print("Training autoencoder...")
    ae, enc, history = train_autoencoder(
        Xtr,
        Xva,
        latent_size=ARCH[inv]["latent"],
        widths=ARCH[inv]["widths"],
        learning_rate=5e-5,
        batch_size=128,
        epochs=30,
        patience=5,
        seed=SEED,
        verbose=0,
    )

    ae_models[inv] = ae
    encoders[inv] = enc

    print(
        f"AE epochs ran: {len(history.history['loss'])}, "
        f"best val loss: {np.min(history.history['val_loss']):.6g}"
    )

    scores_all[inv]["AE"]["test"] = nre_from_model(
        ae,
        Xte,
        batch_size=2048,
    )

    Z_ae_tr = encode_with_model(enc, Xtr, batch_size=2048).astype(np.float32)
    Z_ae_te = encode_with_model(enc, Xte, batch_size=2048).astype(np.float32)

    ae_probe = eval_multinomial_lr_split(
        Z_ae_tr,
        y_train,
        Z_ae_te,
        y_test,
        seed=SEED,
    )

    latent_probe_rows.append({
        "Invariant": inv,
        "Representation": f"AE_d{ARCH[inv]['latent']}",
        "Accuracy": ae_probe["accuracy"],
        "MacroF1": ae_probe["macro_f1"],
        "n_iter": ae_probe["n_iter"],
    })

    print(
        f"AE latent probe: Acc={ae_probe['accuracy']:.3f}, "
        f"Macro-F1={ae_probe['macro_f1']:.3f}"
    )

latent_probe_df = pd.DataFrame(latent_probe_rows)
latent_probe_df.to_csv(OUT_TABLES / "bulk_latent_decoding.csv", index=False)
display(latent_probe_df)

# Save test scores for later scripts/inspection
np.savez(
    OUT_SCORES / "test_reconstruction_scores_seed42.npz",
    Alexander_PCA=scores_all["Alexander"]["PCA"]["test"],
    Alexander_AE=scores_all["Alexander"]["AE"]["test"],
    Jones_PCA=scores_all["Jones"]["PCA"]["test"],
    Jones_AE=scores_all["Jones"]["AE"]["test"],
    HOMFLY_PCA=scores_all["HOMFLY"]["PCA"]["test"],
    HOMFLY_AE=scores_all["HOMFLY"]["AE"]["test"],
)

## 5. Task B: fixed-mass tail enrichment

All tail metrics use exact fixed-mass top-k tails through `compute_tail_table`.

For this test split:

```text
tau = 0.95 -> tail size = 1536
tau = 0.99 -> tail size = 308
```

In [ ]:
from src.evaluation.tail_metrics import compute_tail_table

TAUS_MAIN = [0.95, 0.99]
S_MAIN = [8, 10]

rows = []

for inv in ["Alexander", "Jones", "HOMFLY"]:
    for method in ["PCA", "AE"]:
        sc_test = scores_all[inv][method]["test"]

        df_tail = compute_tail_table(
            scores=sc_test,
            y_signature=y_test,
            s_list=S_MAIN,
            taus=TAUS_MAIN,
        )

        df_tail["Invariant"] = inv
        df_tail["Score"] = f"NRE_{method}"
        df_tail["Split"] = "test"
        df_tail["Scope"] = "MAIN"

        rows.append(df_tail)

df_master = pd.concat(rows, ignore_index=True)

df_master = df_master[
    [
        "Invariant",
        "Score",
        "Split",
        "Scope",
        "Target",
        "Tau",
        "AUROC",
        "AUPRC",
        "Enrichment",
        "Positives",
        "Tail_captured",
        "Tail_size",
        "N",
        "Base_rate",
    ]
].sort_values(["Invariant", "Score", "Target", "Tau"])

display(df_master)
df_master.to_csv(OUT_TABLES / "tail_enrichment_main.csv", index=False)

print("Tail sizes by tau:")
display(df_master.groupby("Tau")["Tail_size"].unique())

## 6. Y12 qualitative stress test

`Y12` is not used for the main quantitative claims because it has only two positives in the held-out test split.

In [ ]:
rows_y12 = []

for method in ["PCA", "AE"]:
    sc_test = scores_all["Jones"][method]["test"]

    df_y12 = compute_tail_table(
        scores=sc_test,
        y_signature=y_test,
        s_list=[12],
        taus=[0.99],
    )

    df_y12["Invariant"] = "Jones"
    df_y12["Score"] = f"NRE_{method}"
    df_y12["Split"] = "test"
    df_y12["Scope"] = "Y12_STRESS"

    rows_y12.append(df_y12)

df_jones_y12 = pd.concat(rows_y12, ignore_index=True)
display(df_jones_y12)

df_jones_y12.to_csv(OUT_TABLES / "jones_Y12_stress.csv", index=False)

## 7. Optional: Jones AE ablation

This is a longer appendix experiment. Enable `RUN_LONG_ABLATION = True` in Section 0 to run it.

It varies latent dimension, seed, and score convention.

In [ ]:
if RUN_LONG_ABLATION:
    from src.evaluation.ablation import run_jones_ae_ablation

    df_ablation = run_jones_ae_ablation(
        X_train=J_tr,
        X_val=J_va,
        X_test=J_te,
        y_test=y_test,
        widths=[128, 64, 64],
        latent_dims=[8, 16, 32],
        seeds=[42, 123, 999],
        score_types=["NRE", "SSE"],
        tau=0.99,
        target_s=10,
        learning_rate=5e-5,
        batch_size=128,
        epochs=30,
        patience=5,
        pred_batch_size=2048,
    )

    df_ablation.to_csv(OUT_TABLES / "jones_ae_ablation_stability.csv", index=False)
    display(df_ablation)
else:
    print("Skipping long ablation. Set RUN_LONG_ABLATION = True to run.")

## 8. Optional: distributional diagnostics

This section fits descriptive tail diagnostics to AE-NRE scores and generates CCDF plots.

These diagnostics are descriptive only; the paper does not claim exact power-law behavior.

In [ ]:
if RUN_DISTRIBUTION_DIAGNOSTICS:
    from src.evaluation.distribution_diagnostics import (
        run_distribution_diagnostics,
        plot_ccdf_row,
    )

    score_dict = {
        "Alexander": scores_all["Alexander"]["AE"]["test"],
        "Jones": scores_all["Jones"]["AE"]["test"],
        "HOMFLY": scores_all["HOMFLY"]["AE"]["test"],
    }

    df_pw, fits = run_distribution_diagnostics(
        score_dict=score_dict,
        sample_max=None,   # For quick testing, use 100000
        n_boot=500,        # For quick testing, use 100
        seed=SEED,
    )

    df_pw.to_csv(OUT_TABLES / "distribution_diagnostics_AE_test_seed42.csv", index=False)
    display(df_pw)

    plot_ccdf_row(
        fits=fits,
        out_pdf=OUT_FIGS / "ccdf_all_invariants_AE_test_seed42.pdf",
        out_png=OUT_FIGS / "ccdf_all_invariants_AE_test_seed42.png",
    )

    try:
        display(Image(filename=str(OUT_FIGS / "ccdf_all_invariants_AE_test_seed42.png")))
    except Exception:
        pass
else:
    print("Skipping distribution diagnostics. Set RUN_DISTRIBUTION_DIAGNOSTICS = True to run.")

## 9. Confounder analysis

This section checks whether simple coefficient-level statistics explain reconstruction tails.

It saves:
- Spearman correlations between AE-NRE and confounders.
- Jones confounder-only rare-event prediction table.
- Tail-overlap table comparing AE tail and confounder-only tail.

In [ ]:
from src.evaluation.confounders import (
    support_width_from_coeffs,
    compute_spearman_confounders,
    build_confounder_matrix_scaled_space,
    evaluate_confounder_models_jones,
    enrichment_sensitivity_stable,
    tail_overlap_against_confounders,
)

crossing_train = metadata_train["number_of_crossings"].to_numpy()
crossing_test = metadata_test["number_of_crossings"].to_numpy()

width_train = metadata_train["maximum_exponent"].to_numpy() - metadata_train["minimum_exponent"].to_numpy()
width_test = metadata_test["maximum_exponent"].to_numpy() - metadata_test["minimum_exponent"].to_numpy()

nre_A = scores_all["Alexander"]["AE"]["test"]
nre_J = scores_all["Jones"]["AE"]["test"]
nre_H = scores_all["HOMFLY"]["AE"]["test"]

width_H_test = support_width_from_coeffs(H_te, eps=1e-8)

conf_A = compute_spearman_confounders(A_te, nre_A, crossing_test, width_test)
conf_A.insert(0, "Invariant", "Alexander")

conf_J = compute_spearman_confounders(J_te, nre_J, crossing_test, width_test)
conf_J.insert(0, "Invariant", "Jones")

conf_H = compute_spearman_confounders(H_te, nre_H, crossing_test, width_H_test)
conf_H.insert(0, "Invariant", "HOMFLY")

conf_all = pd.concat([conf_A, conf_J, conf_H], ignore_index=True)
conf_all["Domain"] = "test"
conf_all["N_test"] = len(metadata_test)

display(conf_all)
conf_all.to_csv(OUT_TABLES / "confounder_spearman.csv", index=False)

### 9.1 Jones confounder-only rare-event prediction

In [ ]:
C_train = build_confounder_matrix_scaled_space(
    X_scaled=J_tr,
    width=width_train,
    crossing=crossing_train,
    l0_tol=0.1,
)

C_test = build_confounder_matrix_scaled_space(
    X_scaled=J_te,
    width=width_test,
    crossing=crossing_test,
    l0_tol=0.1,
)

df_conf_models = evaluate_confounder_models_jones(
    C_train=C_train,
    C_test=C_test,
    sig_train=y_train,
    sig_test=y_test,
    targets=(8, 10),
    taus=(0.95, 0.99),
    seed=SEED,
)

display(df_conf_models)
df_conf_models.to_csv(OUT_TABLES / "confounder_models_jones.csv", index=False)

print("Tail sizes by tau:")
for col in ["Tail@0.95", "Tail@0.99"]:
    if col in df_conf_models.columns:
        print(col, sorted(df_conf_models[col].astype(str).unique())[:5])

### 9.2 Optional: support-width stratification

This produces an auxiliary diagnostic table. It is not necessary for the main paper tables.

In [ ]:
if RUN_SUPPORT_WIDTH_STRATIFICATION:
    def make_df_for_conditioned(signature, crossing, support_width, nre, support_width_type):
        return pd.DataFrame(
            {
                "signature": signature,
                "number_of_crossings": crossing,
                "support_width": support_width,
                "support_width_type": support_width_type,
                "NRE_AE": nre,
            }
        )

    dfA = make_df_for_conditioned(y_test, crossing_test, width_test, nre_A, "exponent_width")
    dfJ = make_df_for_conditioned(y_test, crossing_test, width_test, nre_J, "exponent_width")
    dfH = make_df_for_conditioned(y_test, crossing_test, width_H_test, nre_H, "coef_support_proxy")

    enrich_A = enrichment_sensitivity_stable(dfA)
    enrich_A.insert(0, "Invariant", "Alexander")
    enrich_A["support_width_type"] = "exponent_width"

    enrich_J = enrichment_sensitivity_stable(dfJ)
    enrich_J.insert(0, "Invariant", "Jones")
    enrich_J["support_width_type"] = "exponent_width"

    enrich_H = enrichment_sensitivity_stable(dfH)
    enrich_H.insert(0, "Invariant", "HOMFLY")
    enrich_H["support_width_type"] = "coef_support_proxy"

    enrich_all = pd.concat([enrich_A, enrich_J, enrich_H], ignore_index=True)
    enrich_all["Domain"] = "test"

    display(enrich_all.head(30))
    enrich_all.to_csv(OUT_TABLES / "tail_enrichment_conditioned_test_smoothed.csv", index=False)
else:
    print("Skipping support-width stratification. Set RUN_SUPPORT_WIDTH_STRATIFICATION = True to run.")

### 9.3 AE vs confounder tail overlap

In [ ]:
df_overlap, df_ae_only = tail_overlap_against_confounders(
    ae_scores=nre_J,
    C_train=C_train,
    C_test=C_test,
    sig_train=y_train,
    sig_test=y_test,
    targets=(8, 10),
    taus=(0.95, 0.99),
    seed=SEED,
)

display(df_overlap)
df_overlap.to_csv(OUT_TABLES / "tail_overlap_jones.csv", index=False)

if len(df_ae_only) > 0:
    display(df_ae_only.head(20))
    df_ae_only.to_csv(OUT_TABLES / "ae_only_positive_examples.csv", index=False)

print("Jones Y10, tau=0.99 overlap summary:")
display(
    df_overlap[
        (df_overlap["Target"] == "Y10") & (np.isclose(df_overlap["Tau"], 0.99))
    ][
        [
            "Jaccard",
            "AE_tail_covered_by_Conf",
            "Conf_tail_covered_by_AE",
            "AE_Enrichment",
            "Conf_Enrichment",
            "AE_pos_in_tail",
            "Conf_pos_in_tail",
            "AE_only_tail_size",
            "AE_only_pos_in_tail",
        ]
    ]
)

## 10. Crossing-number controls

This evaluates whether AE-NRE and top-1% tail membership are explained by crossing number.

Tail membership uses exact fixed-mass top-k selection.

In [ ]:
from scipy.stats import spearmanr, kruskal
from sklearn.metrics import mutual_info_score, normalized_mutual_info_score

try:
    from src.evaluation.tail_metrics import topk_tail_mask
except Exception:
    def topk_tail_mask(scores, tau):
        scores = np.asarray(scores, dtype=np.float64).ravel()
        n = len(scores)
        k = int(np.ceil((1.0 - tau) * n))
        order = np.argsort(-scores, kind="mergesort")
        mask = np.zeros(n, dtype=bool)
        mask[order[:k]] = True
        return mask

cross_test = metadata_test["number_of_crossings"].to_numpy().astype(int)

nre_test = {
    "Alexander": np.asarray(scores_all["Alexander"]["AE"]["test"], dtype=np.float64),
    "Jones": np.asarray(scores_all["Jones"]["AE"]["test"], dtype=np.float64),
    "HOMFLY": np.asarray(scores_all["HOMFLY"]["AE"]["test"], dtype=np.float64),
}

TAIL_TAU = 0.99
MIN_GROUP_SIZE = 10

rows_cross = []

for inv, scores in nre_test.items():
    df = pd.DataFrame({
        "cross": cross_test,
        "nre": scores,
    })

    df["is_tail"] = topk_tail_mask(df["nre"].to_numpy(), TAIL_TAU).astype(int)

    rho, rho_p = spearmanr(df["cross"], df["nre"])

    groups = [
        g["nre"].to_numpy()
        for _, g in df.groupby("cross")
        if len(g) >= MIN_GROUP_SIZE
    ]

    if len(groups) >= 2:
        kw_H, kw_p = kruskal(*groups)
    else:
        kw_H, kw_p = np.nan, np.nan

    mi = mutual_info_score(df["cross"], df["is_tail"])
    nmi = normalized_mutual_info_score(df["cross"], df["is_tail"])

    rows_cross.append({
        "Invariant": inv,
        "Domain": "test",
        "N_test": len(df),
        "Tail_tau": TAIL_TAU,
        "Tail_size": int(df["is_tail"].sum()),
        "Spearman_rho_cross_NRE": float(rho),
        "Spearman_p": float(rho_p),
        "Kruskal_H": float(kw_H) if np.isfinite(kw_H) else np.nan,
        "Kruskal_p": float(kw_p) if np.isfinite(kw_p) else np.nan,
        "MI_cross_tail": float(mi),
        "NMI_cross_tail": float(nmi),
        "n_cross_groups": int(df["cross"].nunique()),
    })

df_cross = pd.DataFrame(rows_cross)
display(df_cross)

df_cross.to_csv(OUT_TABLES / "crossing_number_nre_tail_test.csv", index=False)
print("Saved:", OUT_TABLES / "crossing_number_nre_tail_test.csv")

## 11. Optional: OOD crossing-number plot

This section is optional because it expects a precomputed CSV produced by the OOD script.

Expected file:

```text
results/tables/progressive_ood_fixed_train_jones.csv
```

If the file is missing, the notebook skips this plot instead of failing.

In [ ]:
if RUN_OOD_PLOT:
    import matplotlib.pyplot as plt

    OOD_CSV = OUT_TABLES / "progressive_ood_fixed_train_jones.csv"

    if not OOD_CSV.exists():
        print(f"Skipping OOD plot because {OOD_CSV} was not found.")
        print("Run the OOD script first, or place the CSV in results/tables/.")
    else:
        df = pd.read_csv(OOD_CSV)
        df = df.sort_values(["test_cross", "horizon", "k_train_max"]).reset_index(drop=True)

        def plot_line(ax, sub, xcol, ycol, label):
            x = sub[xcol].to_numpy()
            y = sub[ycol].to_numpy()
            ax.plot(x, y, marker="o", linewidth=1, label=label)

        fig = plt.figure(figsize=(7.2, 3.4), dpi=200)
        ax1 = fig.add_subplot(1, 2, 1)
        ax2 = fig.add_subplot(1, 2, 2)

        for h in sorted(df["horizon"].unique()):
            sub = df[df["horizon"] == h]
            plot_line(ax1, sub, "test_cross", "probe_acc", label=f"h={h} acc")
            plot_line(ax1, sub, "test_cross", "probe_macroF1", label=f"h={h} macro-F1")

        ax1.set_xlabel("Test crossing number")
        ax1.set_ylabel("Probe performance (Jones)")
        ax1.set_xticks(sorted(df["test_cross"].unique()))
        ax1.legend(fontsize=7, frameon=False)

        df_auroc = df[np.isfinite(df["pcares_auroc_Y10"].to_numpy())].copy()

        for h in sorted(df_auroc["horizon"].unique()):
            sub = df_auroc[df_auroc["horizon"] == h]
            plot_line(ax2, sub, "test_cross", "pcares_auroc_Y10", label=f"h={h} AUROC")

        ax2.set_xlabel("Test crossing number")
        ax2.set_ylabel(r"PCA-residual AUROC for $Y_{10}$")
        ax2.set_xticks(sorted(df["test_cross"].unique()))
        ax2.legend(fontsize=7, frameon=False)

        plt.tight_layout()

        out_png = OUT_FIGS / "fig_progressive_ood_fixedtrain_jones.png"
        out_pdf = OUT_FIGS / "fig_progressive_ood_fixedtrain_jones.pdf"

        plt.savefig(out_png, dpi=300, bbox_inches="tight")
        plt.savefig(out_pdf, bbox_inches="tight")
        plt.show()

        print("Saved:", out_png)
        print("Saved:", out_pdf)
else:
    print("Skipping OOD plot. Set RUN_OOD_PLOT = True to run.")

## 12. Summary of generated outputs

Core outputs are written to:

```text
results/tables/
results/scores/
results/splits/
results/figures/
```

The script-based pipeline remains the canonical reproduction path; this notebook provides an interactive mirror of the main workflow.